# B4T1 - Diseño de Redes Confiables
## Parte 3: AutoML (Keras Tuner)

Implementación de un proceso de **optimización automática de hiperparámetros** mediante **Keras Tuner** para encontrar la topología más adecuada de la red neuronal. La búsqueda explora distintas configuraciones de:

* Número de capas ocultas.
* Número de neuronas por capa.
* Función de activación.
* Tasa de `dropout`.
* Optimizador.
* Tasa de aprendizaje.

La mejor configuración se selecciona maximizando el **AUC sobre el conjunto de validación**:

$$
\theta^{*}
==========

\underset{\theta \in \Theta}{\operatorname{arg,max}}
;
\operatorname{AUC}*{val}
\left(
f*{\theta}(X_{val}),y_{val}
\right)
$$

donde $\theta$ representa una combinación de hiperparámetros y $\Theta$ es el espacio de búsqueda definido. Para realizar la optimización se utiliza **Hyperband**, que descarta progresivamente las configuraciones menos prometedoras y asigna más recursos a aquellas que obtienen mejores resultados.

Comparamos el **modelo Base**, construido con una arquitectura definida manualmente, frente al **mejor modelo AutoML** encontrado por Keras Tuner. La comparación se realiza sobre el conjunto de test utilizando métricas como `AUC`, `accuracy`, `precision`, `recall` y la función de pérdida.

> **Reutilización**: la carga y el preprocesado de los datos, la **capa customizada de Ratio de Endeudamiento** y la construcción general del modelo se encuentran en `src/base.py`, que actúa como única fuente de verdad y es compartido con `01_base_model.ipynb`. En este apartado no se duplican ni el preprocesado ni los componentes personalizados: Keras Tuner reutiliza el mismo modelo y modifica únicamente los hiperparámetros incluidos en el espacio de búsqueda.


# 1. Imports

In [4]:
import os
import sys
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

import keras
import tensorflow as tf
from keras import ops
import keras_tuner as kt

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
from scipy.stats import pearsonr

# Para poder importar desde src/ aunque el notebook esté en notebooks/
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.base import (
    load_home_credit_data,
    RatioEndeudamientoLayer,
    build_model,
    compute_class_weight_balanced,
    FEATURE_NAMES,
    IDX_GENDER,
)

keras.utils.set_random_seed(SEED)

# 2. Datos

In [5]:
DATA_PATH = os.path.join(ROOT, "data", "application_train.csv")

(X_train, y_train, s_train), (X_test, y_test, s_test) = load_home_credit_data(DATA_PATH)

X_tr, X_val, y_tr, y_val, s_tr, s_val = train_test_split(
    X_train,
    y_train,
    s_train,
    test_size=0.15,
    random_state=SEED,
    stratify=y_train
)

print("X_tr:", X_tr.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("y_tr:", y_tr.shape)
print("s_tr:", s_tr.shape)

X_tr: (209104, 11)
X_val: (36901, 11)
X_test: (61502, 11)
y_tr: (209104,)
s_tr: (209104,)


# 3. Modelo 

El modelo que se utiliza en este apartado parte directamente de lo desarrollado en los apartados anteriores. En concreto, se reutiliza la misma red base, enriquecida con la **capa customizada** diseñada en el apartado 1 y entrenada con la **función de coste FAIR** definida en el apartado 2.

Esto significa que el modelo que ahora se optimiza con AutoML ya incorpora dos elementos clave del trabajo previo:

- La **capa custom** del apartado 1, que introduce dentro de la arquitectura la señal de **ratio de endeudamiento**.
- La **función de coste customizada** del apartado 2, que combina el error de clasificación con una penalización de **aprendizaje justo** para reducir la dependencia respecto a la variable sensible.

Por tanto, en este apartado no se modifica la idea del modelo, sino que se busca su **mejor configuración arquitectónica**. Es decir, se mantienen la lógica del modelo y la FAIR loss, y se optimizan aspectos como el número de capas, el número de neuronas, el dropout y otros hiperparámetros mediante **Keras Tuner**.

In [6]:
SEED = 42
keras.utils.set_random_seed(SEED)

# ------------------------------------------------------------------
# Helpers de etiquetas y class weights
# ------------------------------------------------------------------
def stack_target_sensitive(y, s):
    return np.column_stack([np.asarray(y), np.asarray(s)]).astype("float32")

def compute_class_weight_balanced(y):
    y = np.asarray(y).ravel()
    n = len(y)
    n_pos = y.sum()
    n_neg = n - n_pos
    return {0: n / (2.0 * n_neg), 1: n / (2.0 * n_pos)}

y_tr2 = stack_target_sensitive(y_tr, s_tr)
y_val2 = stack_target_sensitive(y_val, s_val)

cw = compute_class_weight_balanced(y_tr)
W_NEG, W_POS = cw[0], cw[1]

# ------------------------------------------------------------------
# Capa customizada del notebook 1
# ------------------------------------------------------------------
FEATURE_NAMES = [
    "CODE_GENDER",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "DAYS_BIRTH",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "EXT_SOURCE_1_NULL",
    "EXT_SOURCE_2_NULL",
    "EXT_SOURCE_3_NULL",
]

IDX_INCOME = FEATURE_NAMES.index("AMT_INCOME_TOTAL")
IDX_CREDIT = FEATURE_NAMES.index("AMT_CREDIT")
IDX_ANNUITY = FEATURE_NAMES.index("AMT_ANNUITY")

@keras.saving.register_keras_serializable(package="b4t1")
class RatioEndeudamientoLayer(keras.layers.Layer):
    def __init__(
        self,
        idx_income=IDX_INCOME,
        idx_credit=IDX_CREDIT,
        idx_annuity=IDX_ANNUITY,
        w_income=1.0,
        w_credit=1.0,
        w_annuity=1.0,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.idx_income = idx_income
        self.idx_credit = idx_credit
        self.idx_annuity = idx_annuity
        self.w_income = w_income
        self.w_credit = w_credit
        self.w_annuity = w_annuity

    def call(self, inputs):
        income = inputs[:, self.idx_income]
        credit = inputs[:, self.idx_credit]
        annuity = inputs[:, self.idx_annuity]

        carga = (
            self.w_credit * credit
            + self.w_annuity * annuity
            - self.w_income * income
        )

        endeudamiento = ops.tanh(carga)
        endeudamiento = ops.expand_dims(endeudamiento, axis=-1)
        return ops.concatenate([inputs, endeudamiento], axis=-1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1] + 1)

    def get_config(self):
        config = super().get_config()
        config.update({
            "idx_income": self.idx_income,
            "idx_credit": self.idx_credit,
            "idx_annuity": self.idx_annuity,
            "w_income": self.w_income,
            "w_credit": self.w_credit,
            "w_annuity": self.w_annuity,
        })
        return config

# ------------------------------------------------------------------
# FAIR loss del notebook 2
# ------------------------------------------------------------------
EPS = 1e-8

def pearson_abs(p, s, eps=EPS):
    p = ops.cast(ops.reshape(p, (-1,)), "float32")
    s = ops.cast(ops.reshape(s, (-1,)), "float32")
    pm = p - ops.mean(p)
    sm = s - ops.mean(s)
    num = ops.sum(pm * sm)
    den = ops.sqrt(ops.sum(pm ** 2) * ops.sum(sm ** 2)) + eps
    return ops.abs(num / den)

def make_fair_loss(lambda_fair, base_error="bce", w_neg=1.0, w_pos=1.0, eps=EPS):
    lambda_fair = float(lambda_fair)
    assert base_error in ("bce", "mse")

    def fair_loss(y_true, y_pred):
        y = y_true[:, 0:1]
        s = y_true[:, 1:2]
        y_pred = ops.clip(y_pred, 1e-7, 1.0 - 1e-7)

        if base_error == "bce":
            per_sample = keras.losses.binary_crossentropy(y, y_pred)
        else:
            per_sample = ops.mean(ops.square(y - y_pred), axis=-1)

        w = y[:, 0] * w_pos + (1.0 - y[:, 0]) * w_neg
        base = ops.sum(w * per_sample) / (ops.sum(w) + eps)

        corr = pearson_abs(y_pred, s, eps)
        return base + lambda_fair * corr

    fair_loss.__name__ = f"fair_loss_lambda_{lambda_fair:g}"
    return fair_loss

# ------------------------------------------------------------------
# Métricas del notebook 2
# ------------------------------------------------------------------
class FairCorr(keras.metrics.Metric):
    def __init__(self, name="fair_corr", eps=EPS, **kwargs):
        super().__init__(name=name, **kwargs)
        self.eps = eps
        self.n   = self.add_weight(name="n", initializer="zeros")
        self.sp  = self.add_weight(name="sp", initializer="zeros")
        self.ss  = self.add_weight(name="ss", initializer="zeros")
        self.spp = self.add_weight(name="spp", initializer="zeros")
        self.sss = self.add_weight(name="sss", initializer="zeros")
        self.sps = self.add_weight(name="sps", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        p = ops.reshape(ops.cast(y_pred, "float32"), (-1,))
        s = ops.reshape(ops.cast(y_true[:, 1:2], "float32"), (-1,))
        self.n.assign_add(ops.cast(ops.size(p), "float32"))
        self.sp.assign_add(ops.sum(p))
        self.ss.assign_add(ops.sum(s))
        self.spp.assign_add(ops.sum(p * p))
        self.sss.assign_add(ops.sum(s * s))
        self.sps.assign_add(ops.sum(p * s))

    def result(self):
        cov = self.n * self.sps - self.sp * self.ss
        vp = self.n * self.spp - self.sp ** 2
        vs = self.n * self.sss - self.ss ** 2
        return ops.abs(cov / (ops.sqrt(vp * vs) + self.eps))

    def reset_state(self):
        for v in self.variables:
            v.assign(0.0)

class TargetAUC(keras.metrics.AUC):
    def update_state(self, y_true, y_pred, sample_weight=None):
        return super().update_state(y_true[:, 0:1], y_pred, sample_weight)

# ------------------------------------------------------------------
# Modelo completo: capa custom + arquitectura + FAIR loss
# ------------------------------------------------------------------
def build_fair_model(
    input_dim,
    lambda_fair=0.5,
    units1=64,
    units2=32,
    dropout=0.2,
    learning_rate=1e-3
):
    inputs = keras.Input(shape=(input_dim,), name="features")
    x = RatioEndeudamientoLayer(name="ratio_endeudamiento")(inputs)
    x = keras.layers.Dense(units1, activation="relu")(x)
    x = keras.layers.Dropout(dropout)(x)
    x = keras.layers.Dense(units2, activation="relu")(x)
    x = keras.layers.Dropout(dropout)(x)
    outputs = keras.layers.Dense(1, activation="sigmoid", name="pd")(x)

    model = keras.Model(inputs, outputs, name="credit_model_fair")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss=make_fair_loss(lambda_fair, "bce", W_NEG, W_POS),
        metrics=[TargetAUC(name="auc"), FairCorr()],
    )
    return model

model = build_fair_model(input_dim=X_tr.shape[1])
model.summary()


Model: "credit_model_fair"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ features (InputLayer)           │ (None, 11)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ratio_endeudamiento             │ (None, 12)             │             0 │
│ (RatioEndeudamientoLayer)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pd (Dense)                      │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

# 4. Implementación Keras-tuner

En este bloque se utiliza **Keras Tuner** para automatizar la búsqueda de la mejor configuración del modelo. La idea es no fijar manualmente una única arquitectura, sino dejar que el algoritmo pruebe distintas combinaciones de hiperparámetros y seleccione la que mejor funciona en validación.

La función `build_tuner_model(hp)` define el espacio de búsqueda. Para cada prueba (*trial*), Keras Tuner construye un modelo cambiando varios elementos:

- el número de capas ocultas (`n_hidden`)
- la función de activación (`relu`, `tanh` o `elu`)
- el número de neuronas de cada capa (`32`, `64` o `128`)
- el porcentaje de `dropout` en cada capa
- el valor de `lambda_fair` de la función de coste justa
- la tasa de aprendizaje del optimizador Adam

De esta forma, cada *trial* corresponde a una combinación distinta de arquitectura y entrenamiento. Después, `kt.RandomSearch` evalúa hasta **50 configuraciones diferentes**, utilizando como criterio principal la métrica **`val_auc`**, es decir, el AUC sobre el conjunto de validación.

Además, se aplica **Early Stopping** para evitar entrenamientos innecesarios cuando el modelo deja de mejorar en validación. Una vez terminada la búsqueda, Keras Tuner devuelve:

- `best_hp`: los mejores hiperparámetros encontrados
- `best_model`: el modelo construido con esa configuración óptima

En resumen, Keras Tuner actúa como un procedimiento de **AutoML**, explorando automáticamente distintas arquitecturas y seleccionando la que ofrece el mejor equilibrio de rendimiento en validación dentro del espacio de búsqueda definido.

In [ ]:
KT_DIR = os.path.join(ROOT, "outputs", "kt_automl")
os.makedirs(KT_DIR, exist_ok=True)

def build_tuner_model(hp):
    keras.utils.set_random_seed(SEED)

    inputs = keras.Input(shape=(X_tr.shape[1],), name="features")
    x = RatioEndeudamientoLayer(name="ratio_endeudamiento")(inputs)

    n_hidden = hp.Int("n_hidden", min_value=1, max_value=3, step=1)
    activation = hp.Choice("activation", ["relu", "tanh", "elu"])

    for i in range(n_hidden):
        units = hp.Choice(f"units_{i+1}", [32, 64, 128])
        dropout = hp.Float(f"dropout_{i+1}", min_value=0.0, max_value=0.5, step=0.1)
        x = keras.layers.Dense(units, activation=activation, name=f"dense_{i+1}")(x)
        x = keras.layers.Dropout(dropout, name=f"dropout_{i+1}")(x)

    outputs = keras.layers.Dense(1, activation="sigmoid", name="pd")(x)

    model = keras.Model(inputs, outputs, name="credit_model_fair_tunable")

    lambda_fair = hp.Choice("lambda_fair", [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0])
    learning_rate = hp.Choice("learning_rate", [1e-2, 1e-3, 1e-4])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss=make_fair_loss(lambda_fair, "bce", W_NEG, W_POS),
        metrics=[TargetAUC(name="auc"), FairCorr()],
    )
    return model

tuner = kt.RandomSearch(
    build_tuner_model,
    objective=kt.Objective("val_auc", direction="max"),
    max_trials=50,
    seed=SEED,
    overwrite=True,
    directory=KT_DIR,
    project_name="fair_topology_search",
)

es = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
)

t0 = time.time()
tuner.search(
    X_tr,
    y_tr2,
    validation_data=(X_val, y_val2),
    epochs=15,
    batch_size=512,
    callbacks=[es],
    verbose=1,
)
print(f"Búsqueda completada en {time.time() - t0:.0f}s")

best_hp = tuner.get_best_hyperparameters(1)[0]
best_model = tuner.get_best_models(1)[0]

print("Mejores hiperparámetros:")
for k, v in best_hp.values.items():
    print(f"  {k}: {v}")

best_model.summary()

Trial 50 Complete [00h 00m 11s]
val_auc: 0.717918872833252

Best val_auc So Far: 0.7367756366729736
Total elapsed time: 00h 08m 11s
Búsqueda completada en 491s
Mejores hiperparámetros:
  n_hidden: 3
  activation: relu
  units_1: 32
  dropout_1: 0.1
  lambda_fair: 0.01
  learning_rate: 0.01
  units_2: 64
  dropout_2: 0.2
  units_3: 32
  dropout_3: 0.0


Model: "credit_model_fair_tunable"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ features (InputLayer)           │ (None, 11)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ratio_endeudamiento             │ (None, 12)             │             0 │
│ (RatioEndeudamientoLayer)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pd (Dense)                      │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,641 (18.13 KB)

 Trainable params: 4,641 (18.13 KB)

 Non-trainable params: 0 (0.00 B)

# 5. Resultado 

La búsqueda automática con Keras Tuner evaluó **50 configuraciones distintas** y seleccionó como mejor modelo el que obtuvo un **AUC de validación de 0.7368**. Este resultado indica que, dentro del espacio de búsqueda definido, la arquitectura elegida fue la que mejor capacidad discriminativa mostró sobre el conjunto de validación.

La configuración óptima encontrada corresponde a una red con **3 capas ocultas**, con estructura **32-64-32**, activación **ReLU** y valores de `dropout` moderados (`0.1`, `0.2` y `0.0`). Además, el mejor valor de la penalización de justicia fue **`lambda_fair = 0.01`**, lo que sugiere que en esta búsqueda el mejor equilibrio entre rendimiento predictivo y restricción FAIR se alcanzó con una penalización relativamente baja.

En conjunto, el resultado muestra que la arquitectura más adecuada no es ni la más pequeña ni la más grande de las probadas, sino una topología intermedia, suficientemente flexible para captar patrones del problema sin introducir una complejidad excesiva.